In [1]:
from pathlib import Path
import importlib
import sys

import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, LogLocator, MultipleLocator
import numpy as np
import pandas as pd
from IPython.display import display

NB_PATH = Path.cwd()
REPO_ROOT = NB_PATH
for candidate in [NB_PATH, *NB_PATH.parents]:
    if (candidate / 'data').exists() and (candidate / 'analysis').exists():
        REPO_ROOT = candidate
        break

sys.path.insert(0, str(REPO_ROOT / 'analysis' / 'src'))
import orca
import orca.leaks as leak_tools

importlib.reload(leak_tools)
importlib.reload(orca)

RAW_DIR = REPO_ROOT / 'data' / 'raw'
TC_CALIBRATION_PATH = REPO_ROOT / 'data' / 'processed' / 'calibration' / 'TC_calibration_20260420.csv'

plt.rcParams.update({
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'font.size': 13,
})

print(f'Repo root: {REPO_ROOT}')
print(f'TC calibration: {TC_CALIBRATION_PATH}')

Repo root: /home/aamy/Documents/hfe-system
TC calibration: /home/aamy/Documents/hfe-system/data/processed/calibration/TC_calibration_20260420.csv


In [2]:
SYSTEM_POST_FIX_SOURCE_LOG = RAW_DIR / 'leak_test' / 'tc_log_20260319_115916.csv'
SYSTEM_POST_FIX_LOG = REPO_ROOT / 'data' / 'processed' / 'leak_test' / 'log_20260319_post_peak_after_leak_fixes.csv'
APRIL_01_SOURCE_LOG = RAW_DIR / 'leak_test' / 'log_20260401_105911.csv'
APRIL_01_POST_PEAK_LOG = REPO_ROOT / 'data' / 'processed' / 'leak_test' / 'log_20260401_decay.csv'
APRIL_14_SOURCE_LOG = RAW_DIR / 'leak_test' / 'log_20260414_173423.csv'
APRIL_14_POST_PEAK_LOG = REPO_ROOT / 'data' / 'processed' / 'leak_test' / 'log_20260414_decay.csv'
APRIL_20_SOURCE_LOG = RAW_DIR / 'log_20260420_125040.csv'
APRIL_20_POST_PEAK_LOG = REPO_ROOT / 'data' / 'processed' / 'leak_test' / 'log_20260420_decay.csv'
APRIL_23_SOURCE_LOG = RAW_DIR / 'leak_test' / 'log_20260423_172104.csv'
APRIL_23_POST_PEAK_LOG = REPO_ROOT / 'data' / 'processed' / 'leak_test' / 'log_20260423_decay.csv'
MARCH_13_SOURCE_LOG = RAW_DIR / 'leak_test' / 'log_20260313_145244.csv'
MARCH_13_POST_PEAK_LOG = REPO_ROOT / 'data' / 'processed' / 'leak_test' / 'log_20260313_decay.csv'


def backfill_pressure_gauge_columns(frame):
    for abs_column, gauge_column in [
        ('pump_pressure_before_bar_abs', 'pump_pressure_before_bar'),
        ('pump_pressure_after_bar_abs', 'pump_pressure_after_bar'),
        (leak_tools.PRESSURE_ABS_COLUMN, leak_tools.PRESSURE_GAUGE_COLUMN),
    ]:
        if abs_column in frame.columns and gauge_column not in frame.columns:
            frame[gauge_column] = pd.to_numeric(frame[abs_column], errors='coerce') - leak_tools.P_ATM_BAR
    return frame


def export_system_log_slice(frame, output_path):
    sliced_frame = backfill_pressure_gauge_columns(frame.copy())
    sliced_frame[leak_tools.TIME_COLUMN] = pd.to_numeric(
        sliced_frame[leak_tools.TIME_COLUMN],
        errors='coerce',
    )
    sliced_frame = sliced_frame.dropna(subset=[leak_tools.TIME_COLUMN]).copy()
    sliced_frame[leak_tools.TIME_COLUMN] = (
        sliced_frame[leak_tools.TIME_COLUMN] - float(sliced_frame[leak_tools.TIME_COLUMN].iloc[0])
    )
    output_path.parent.mkdir(parents=True, exist_ok=True)
    sliced_frame.to_csv(output_path, index=False)
    return output_path


def build_post_peak_system_log(source_path, output_path, *, pressure_column=leak_tools.PRESSURE_ABS_COLUMN):
    frame = pd.read_csv(source_path, comment='#').dropna(subset=[leak_tools.TIME_COLUMN, pressure_column]).copy()
    peak_index = int(pd.to_numeric(frame[pressure_column], errors='coerce').idxmax())
    return export_system_log_slice(frame.iloc[peak_index:].copy(), output_path)


build_post_peak_system_log(SYSTEM_POST_FIX_SOURCE_LOG, SYSTEM_POST_FIX_LOG)
build_post_peak_system_log(MARCH_13_SOURCE_LOG, MARCH_13_POST_PEAK_LOG)
build_post_peak_system_log(APRIL_01_SOURCE_LOG, APRIL_01_POST_PEAK_LOG)
build_post_peak_system_log(APRIL_14_SOURCE_LOG, APRIL_14_POST_PEAK_LOG)
build_post_peak_system_log(APRIL_20_SOURCE_LOG, APRIL_20_POST_PEAK_LOG)
build_post_peak_system_log(APRIL_23_SOURCE_LOG, APRIL_23_POST_PEAK_LOG)

SYSTEM_PRESSURE_LOGS = [
    RAW_DIR / 'leak_test' / 'log_20260311_114721.csv',
    RAW_DIR / 'leak_test' / 'log_20260312_172440.csv',
    MARCH_13_POST_PEAK_LOG,
    SYSTEM_POST_FIX_LOG,
    APRIL_01_POST_PEAK_LOG,
    APRIL_14_POST_PEAK_LOG,
    APRIL_20_POST_PEAK_LOG,
    APRIL_23_POST_PEAK_LOG,
]

system_results = [leak_tools.analyze_system_pressure_log(path) for path in SYSTEM_PRESSURE_LOGS]
system_result_by_log_file = {result.series.csv_path.name: result for result in system_results}
system_summary = leak_tools.system_pressure_summary_table(system_results)

top_system_result = system_result_by_log_file['log_20260312_172440.csv']
march_13_system_result = system_result_by_log_file['log_20260313_decay.csv']
post_fix_system_result = system_result_by_log_file['log_20260319_post_peak_after_leak_fixes.csv']
april_01_summary_result = system_result_by_log_file['log_20260401_decay.csv']
april_14_summary_result = system_result_by_log_file['log_20260414_decay.csv']
april_20_summary_result = system_result_by_log_file['log_20260420_decay.csv']
april_23_summary_result = system_result_by_log_file['log_20260423_decay.csv']


PLOT_X_MAX_H = 15.0
MIN_LOG_PRESSURE_BAR = 0.003
TAIL_PRESSURE_CUT_BAR = 0.05
FIT_RESOLVED_MIN_PRESSURE_BAR = TAIL_PRESSURE_CUT_BAR
FIT_DISPLAY_MIN_PRESSURE_BAR = TAIL_PRESSURE_CUT_BAR


def pressure_above_atm_bar(pressure_abs_bar):
    return np.maximum(np.asarray(pressure_abs_bar, dtype=float) - leak_tools.P_ATM_BAR, MIN_LOG_PRESSURE_BAR)


def pressure_above_atm_yerr(pressure_abs_bar, sigma_bar):
    pressure_abs_bar = np.asarray(pressure_abs_bar, dtype=float)
    sigma_bar = np.asarray(sigma_bar, dtype=float)
    y = pressure_above_atm_bar(pressure_abs_bar)
    y_low = pressure_above_atm_bar(pressure_abs_bar - sigma_bar)
    y_high = pressure_above_atm_bar(pressure_abs_bar + sigma_bar)
    return np.vstack([np.maximum(y - y_low, 0.0), np.maximum(y_high - y, 0.0)])


def format_scientific_pm(value, error, *, precision=2):
    if value == 0.0:
        exponent = 0
    else:
        exponent = int(np.floor(np.log10(abs(value))))
    scale = 10.0 ** exponent
    scaled_value = value / scale
    scaled_error = error / scale
    if exponent == 0:
        return rf'({scaled_value:.{precision}f} \pm {scaled_error:.{precision}f})'
    return rf'({scaled_value:.{precision}f} \pm {scaled_error:.{precision}f}) \times 10^{{{exponent}}}'


def resolved_fixed_tail_fit(result, *, min_pressure_above_atm_bar=FIT_RESOLVED_MIN_PRESSURE_BAR):
    resolved_mask = (
        np.asarray(result.averaged.pressure_abs_bar, dtype=float) - leak_tools.P_ATM_BAR
        >= min_pressure_above_atm_bar
    )
    if np.count_nonzero(resolved_mask) < 3:
        return None

    curve_time_h = np.linspace(
        0.0,
        min(float(result.averaged.time_h[-1]), PLOT_X_MAX_H),
        500,
    )
    fixed_fit, curve_pressure_bar, _ = leak_tools.fit_fixed_tail_exponential_with_band(
        result.averaged.time_h[resolved_mask],
        result.averaged.pressure_abs_bar[resolved_mask],
        result.averaged.pressure_sigma_bar[resolved_mask],
        curve_time_h,
        asymptote_bar=leak_tools.P_ATM_BAR,
    )
    displayed_mask = curve_pressure_bar - leak_tools.P_ATM_BAR >= FIT_DISPLAY_MIN_PRESSURE_BAR
    fit_at_data = leak_tools.fixed_tail_exponential_model(
        result.averaged.time_h[resolved_mask],
        fixed_fit.amplitude_bar,
        fixed_fit.k_per_h,
        asymptote_bar=fixed_fit.asymptote_bar,
    )
    rmse_mbar = 1000.0 * leak_tools.rmse_bar(
        result.averaged.pressure_abs_bar[resolved_mask],
        fit_at_data,
    )
    selected_fit = leak_tools.fixed_tail_fit_to_exponential_fit(fixed_fit)
    selected_leak = leak_tools.compute_hfe_equivalent_leak(
        selected_fit.k_per_h,
        selected_fit.k_err_per_h,
        result.top_gas_trap.hfe_vapor_pressure_abs_bar,
        hfe_temp_c=result.series.room_temp_c,
    )
    return {
        'fixed_tail_fit': fixed_fit,
        'fit': selected_fit,
        'leak': selected_leak,
        'fit_curve_time_h': curve_time_h[displayed_mask],
        'fit_curve_pressure_bar': curve_pressure_bar[displayed_mask],
        'fit_mask': resolved_mask,
        'rmse_mbar': rmse_mbar,
        'note': (
            rf'fit uses P-P_atm >= {(min_pressure_above_atm_bar * 1000.0):.0f} mbar; '
            rf'tail below this threshold is excluded'
        ),
    }


def trace_fit_curve(trace):
    override = trace.get('fit_override')
    if override is not None:
        return override['fit_curve_time_h'], override['fit_curve_pressure_bar']
    result = trace['result']
    return result.fit_curve_time_h, result.fit_curve_pressure_bar


def trace_selected_fit(trace):
    override = trace.get('fit_override')
    if override is not None:
        return override['fit'], override['leak'], override['rmse_mbar'], override.get('note'), override.get('fit_mask')
    result = trace['result']
    return result.fit, result.leak, result.rmse_mbar, None, None


def system_pressure_drop_metrics(result, fit_override=None):
    if fit_override is not None:
        return leak_tools.compute_start_decay_metrics(fit_override['fixed_tail_fit'])
    if result.fit is None:
        return None, None
    fixed_fit, _, _ = leak_tools.fit_fixed_tail_exponential_with_band(
        result.averaged.time_h,
        result.averaged.pressure_abs_bar,
        result.averaged.pressure_sigma_bar,
        result.fit_curve_time_h if result.fit_curve_time_h is not None else result.averaged.time_h,
        asymptote_bar=result.fit.asymptote_bar,
    )
    return leak_tools.compute_start_decay_metrics(fixed_fit)


def system_fit_row(label, result, update_note, fit_override=None):
    pressure_drop = '—'
    hfe_loss = '—'
    exp_fit = '—'
    drop_mbar_h, drop_err_mbar_h = system_pressure_drop_metrics(result, fit_override)
    if drop_mbar_h is not None:
        pressure_drop = f'{drop_mbar_h:.2f} ± {drop_err_mbar_h:.2f}'
    selected_fit = fit_override['fit'] if fit_override is not None else result.fit
    selected_leak = fit_override['leak'] if fit_override is not None else result.leak
    if selected_leak is not None:
        hfe_loss = leak_tools.format_leak_annotation_line(selected_leak, unit='L/year', use_hfe_loss=True)
    if selected_fit is not None:
        initial_pressure_bar = selected_fit.asymptote_bar + selected_fit.amplitude_bar
        k_text = format_scientific_pm(selected_fit.k_per_h, selected_fit.k_err_per_h, precision=2)
        exp_fit = (
            rf'P$_{{\infty}}$ = {selected_fit.asymptote_bar:.3f} bar, '
            rf'P$_0$ = {initial_pressure_bar:.3f} bar, '
            rf'k = ${k_text}\,\mathrm{{h}}^{{-1}}$'
        )
    return [label, update_note, pressure_drop, hfe_loss, exp_fit]


def plot_system_pressure_summary(traces, *, source_note=None):
    fig, axis = plt.subplots(figsize=(12, 7.8))
    y_samples = []

    for trace in traces:
        result = trace['result']
        y = pressure_above_atm_bar(result.averaged.pressure_abs_bar)
        y_low = pressure_above_atm_bar(result.averaged.pressure_abs_bar - result.averaged.pressure_sigma_bar)
        y_high = pressure_above_atm_bar(result.averaged.pressure_abs_bar + result.averaged.pressure_sigma_bar)
        axis.fill_between(
            result.averaged.time_h,
            y_low,
            y_high,
            color=trace['color'],
            alpha=trace.get('alpha', 0.18),
            label=trace.get('band_label', '_nolegend_'),
        )
        axis.plot(
            result.averaged.time_h,
            y,
            color=trace['color'],
            linewidth=3.0,
            label=trace['line_label'],
        )
        fit_curve_time_h, fit_curve_pressure_bar = trace_fit_curve(trace)
        if fit_curve_time_h is not None and fit_curve_pressure_bar is not None and len(fit_curve_time_h) > 0:
            fit_y = pressure_above_atm_bar(fit_curve_pressure_bar)
            axis.plot(
                fit_curve_time_h,
                fit_y,
                color='black',
                linewidth=1.25,
                linestyle=(0, (6, 3)),
                alpha=0.80,
                zorder=6,
                label=trace.get('fit_label', '_nolegend_'),
            )
            y_samples.append(fit_y)
        y_samples.extend([y_low, y_high])

    y_all = np.concatenate([values[np.isfinite(values) & (values > 0.0)] for values in y_samples])
    y_min = float(np.min(y_all))
    y_max = float(np.max(y_all))

    axis.set_xlim(0.0, PLOT_X_MAX_H)
    axis.set_yscale('log')
    axis.set_ylim(max(MIN_LOG_PRESSURE_BAR, y_min / 1.25), y_max * 1.25)
    axis.set_xlabel('Time (hours)', fontsize=20, labelpad=12)
    axis.set_ylabel('Pressure (barg)', fontsize=20)
    axis.tick_params(axis='both', which='major', labelsize=16)
    axis.tick_params(axis='both', which='minor', labelsize=12)
    axis.xaxis.set_major_locator(MultipleLocator(2.5))
    axis.xaxis.set_minor_locator(MultipleLocator(0.5))
    axis.yaxis.set_major_locator(LogLocator(base=10.0, subs=(1.0,), numticks=8))
    axis.yaxis.set_minor_locator(LogLocator(base=10.0, subs=np.arange(2, 10) * 0.1, numticks=80))
    axis.set_axisbelow(True)
    axis.grid(True, which='major', alpha=0.36, linewidth=0.8)
    axis.grid(True, which='minor', alpha=0.18, linewidth=0.55, linestyle=':')
    axis.legend(
        loc='lower right',
        bbox_to_anchor=(0.99, 0.08),
        fontsize=15,
        framealpha=0.95,
        ncol=2,
        columnspacing=1.1,
    )

    fig.subplots_adjust(left=0.10, right=0.98, top=0.97, bottom=0.13)
    if source_note is not None:
        fig.text(
            0.99,
            0.05,
            source_note,
            ha='right',
            va='bottom',
            fontsize=9,
            color='0.45',
        )
    return fig, axis


high_leak_fit_overrides = {
    'Initial check': resolved_fixed_tail_fit(top_system_result),
    'FLM bolts retorqued': resolved_fixed_tail_fit(march_13_system_result),
}

system_summary_traces = [
    {
        'result': top_system_result,
        'color': '#D55E00',
        'line_label': 'Initial check',
        'fit_override': high_leak_fit_overrides['Initial check'],
    },
    {
        'result': march_13_system_result,
        'color': '#E69F00',
        'line_label': 'FLM bolts retorqued',
        'fit_override': high_leak_fit_overrides['FLM bolts retorqued'],
    },
    {
        'result': post_fix_system_result,
        'color': '#CC6677',
        'line_label': 'GTP NPT resealed',
    },
    {
        'result': april_01_summary_result,
        'color': '#56B4E9',
        'line_label': 'RTL rebent',
    },
    {
        'result': april_14_summary_result,
        'color': '#0072B2',
        'line_label': 'TC calibration',
    },
    {
        'result': april_23_summary_result,
        'color': '#009E73',
        'line_label': 'Pump PRV installed',
    },
]
system_fit_rows = [
    system_fit_row('Initial check', top_system_result, 'First check', high_leak_fit_overrides['Initial check']),
    system_fit_row('FLM bolts retorqued', march_13_system_result, 'FLM bolts retorqued', high_leak_fit_overrides['FLM bolts retorqued']),
    system_fit_row('GTP NPT resealed', post_fix_system_result, 'GTP NPT resealed'),
    system_fit_row('RTL rebent', april_01_summary_result, 'RTL rebent'),
    system_fit_row('TC calibration', april_14_summary_result, 'TCs calibration'),
    system_fit_row('Pump PRV installed', april_23_summary_result, 'Pump PRV installed'),
]

system_fit_table = pd.DataFrame(
    system_fit_rows,
    columns=[
        'Dataset',
        'System update done',
        'Pressure drop (mbar/h)',
        'HFE-7200 loss',
        'Exponential parameters',
    ],
)


def system_fit_diagnostic_row(label, result, fit_override=None):
    selected_fit = fit_override['fit'] if fit_override is not None else result.fit
    selected_rmse_mbar = fit_override['rmse_mbar'] if fit_override is not None else result.rmse_mbar
    diagnostic_mask = fit_override['fit_mask'] if fit_override is not None else np.ones_like(result.averaged.time_h, dtype=bool)
    if selected_fit is None:
        return {
            'Dataset': label,
            'P_inf_bar_abs': np.nan,
            'RMSE_mbar': np.nan,
            'first_residual_mbar': np.nan,
            'last_residual_mbar': np.nan,
            'max_abs_residual_mbar': np.nan,
            'note': result.warning or 'no fit',
        }
    fit_at_data = leak_tools.fixed_tail_exponential_model(
        result.averaged.time_h[diagnostic_mask],
        selected_fit.amplitude_bar,
        selected_fit.k_per_h,
        asymptote_bar=selected_fit.asymptote_bar,
    )
    residual_mbar = 1000.0 * (result.averaged.pressure_abs_bar[diagnostic_mask] - fit_at_data)
    note = fit_override['note'] if fit_override is not None else (
        'not single-exponential over full range' if float(selected_rmse_mbar or 0.0) > 25.0 else 'ok'
    )
    return {
        'Dataset': label,
        'P_inf_bar_abs': selected_fit.asymptote_bar,
        'RMSE_mbar': selected_rmse_mbar,
        'first_residual_mbar': residual_mbar[0],
        'last_residual_mbar': residual_mbar[-1],
        'max_abs_residual_mbar': np.max(np.abs(residual_mbar)),
        'note': note,
    }


system_fit_diagnostics = pd.DataFrame([
    system_fit_diagnostic_row('Initial check', top_system_result, high_leak_fit_overrides['Initial check']),
    system_fit_diagnostic_row('FLM bolts retorqued', march_13_system_result, high_leak_fit_overrides['FLM bolts retorqued']),
    system_fit_diagnostic_row('GTP NPT resealed', post_fix_system_result),
    system_fit_diagnostic_row('RTL rebent', april_01_summary_result),
    system_fit_diagnostic_row('TC calibration', april_14_summary_result),
    system_fit_diagnostic_row('Pump PRV installed', april_23_summary_result),
])

fig, axis = plot_system_pressure_summary(system_summary_traces)

tank_gauge_psi = (top_system_result.averaged.pressure_abs_bar - leak_tools.P_ATM_BAR) * 14.5037738
alignment_order = np.argsort(tank_gauge_psi)
top_system_start_time_h = np.interp(
    24.0,
    tank_gauge_psi[alignment_order],
    top_system_result.averaged.time_h[alignment_order],
)
top_system_time_h = top_system_start_time_h + np.array([0.0, 1.0, 2.0], dtype=float)
top_system_gauge_psi = np.array([24.0, 10.0, 4.1], dtype=float)
top_system_pressure_bar_abs = leak_tools.P_ATM_BAR + top_system_gauge_psi / 14.5037738

top_gauge_min_psi = -30.0 * 0.491154
top_gauge_max_psi = 60.0
top_gauge_span_psi = top_gauge_max_psi - top_gauge_min_psi
top_gauge_quarter_psi = 0.25 * top_gauge_span_psi


def top_gauge_sigma_psi(reading_psig: float) -> float:
    in_end_quarter = (
        reading_psig <= top_gauge_min_psi + top_gauge_quarter_psi
        or reading_psig >= top_gauge_max_psi - top_gauge_quarter_psi
    )
    accuracy_fraction = 0.02 if in_end_quarter else 0.01
    return accuracy_fraction * top_gauge_span_psi / np.sqrt(3.0)


top_system_sigma_bar = np.array(
    [top_gauge_sigma_psi(reading) / 14.5037738 for reading in top_system_gauge_psi],
    dtype=float,
)
bottom_system_start_time_h = top_system_result.averaged.time_h[0]
bottom_system_time_h = bottom_system_start_time_h + np.array([0.0, 1.0, 2.0], dtype=float)
bottom_system_pressure_bar_abs = np.array([2.75, 2.75, 2.75], dtype=float)
bottom_system_sigma_bar = np.full(3, 0.1, dtype=float)
full_system_no_gas_trap_time_h = bottom_system_start_time_h + np.array([0.0, 1.0, 2.0], dtype=float)
full_system_no_gas_trap_pressure_bar_abs = np.array([2.78, 2.78, 2.75], dtype=float)
full_system_no_gas_trap_sigma_bar = np.full(3, 0.1, dtype=float)
gas_trap_gauge_bar = np.array([1.8, 1.78, 1.76, 1.76], dtype=float)
gas_trap_gauge_psi = gas_trap_gauge_bar * 14.5037738
gas_trap_start_time_h = np.interp(
    gas_trap_gauge_psi[0],
    tank_gauge_psi[alignment_order],
    top_system_result.averaged.time_h[alignment_order],
)
gas_trap_time_h = gas_trap_start_time_h + np.array([0.0, 1.0, 2.0, 3.0], dtype=float)
gas_trap_pressure_bar_abs = leak_tools.P_ATM_BAR + gas_trap_gauge_bar
gas_trap_sigma_bar = np.array(
    [top_gauge_sigma_psi(reading) / 14.5037738 for reading in gas_trap_gauge_psi],
    dtype=float,
)

axis.errorbar(
    top_system_time_h,
    pressure_above_atm_bar(top_system_pressure_bar_abs),
    yerr=pressure_above_atm_yerr(top_system_pressure_bar_abs, top_system_sigma_bar),
    fmt='o',
    capsize=4,
    color='#882255',
    label='Top of system',
)
axis.errorbar(
    bottom_system_time_h,
    pressure_above_atm_bar(bottom_system_pressure_bar_abs),
    yerr=pressure_above_atm_yerr(bottom_system_pressure_bar_abs, bottom_system_sigma_bar),
    fmt='o',
    capsize=4,
    color='#44AA99',
    label='Bottom of system',
)
axis.errorbar(
    full_system_no_gas_trap_time_h,
    pressure_above_atm_bar(full_system_no_gas_trap_pressure_bar_abs),
    yerr=pressure_above_atm_yerr(full_system_no_gas_trap_pressure_bar_abs, full_system_no_gas_trap_sigma_bar),
    fmt='o',
    capsize=4,
    color='#332288',
    label='Without gas trap',
)
current_xmin, current_xmax = axis.get_xlim()
current_ymin, current_ymax = axis.get_ylim()
y_measurement_max = max(
    float(np.max(pressure_above_atm_bar(top_system_pressure_bar_abs + top_system_sigma_bar))),
    float(np.max(pressure_above_atm_bar(bottom_system_pressure_bar_abs + bottom_system_sigma_bar))),
    float(np.max(pressure_above_atm_bar(full_system_no_gas_trap_pressure_bar_abs + full_system_no_gas_trap_sigma_bar))),
)
axis.set_xlim(0.0, PLOT_X_MAX_H)
axis.set_ylim(max(MIN_LOG_PRESSURE_BAR, current_ymin), max(current_ymax, y_measurement_max * 1.10))
axis.legend(
    loc='lower right',
    bbox_to_anchor=(0.99, 0.08),
    fontsize=14,
    framealpha=0.95,
    ncol=2,
    columnspacing=1.1,
)

display(fig)
plt.close(fig)
display(system_fit_table)
print('Fit diagnostics; early high-leak rows use the resolved-range fit and omit the pressure-readout floor from the residuals.')
display(system_fit_diagnostics.round({
    'P_inf_bar_abs': 5,
    'RMSE_mbar': 1,
    'first_residual_mbar': 1,
    'last_residual_mbar': 1,
    'max_abs_residual_mbar': 1,
}))

<Figure size 1800x1170 with 1 Axes>

,Dataset,System update done,Pressure drop (mbar/h),HFE-7200 loss,Exponential parameters
0,Initial check,First check,767.50 ± 8.16,$(2.49 \pm 0.02) \times 10^{1}$ L/year,"P$_{\infty}$ = 1.013 bar, P$_0$ = 2.684 bar, k..."
1,FLM bolts retorqued,FLM bolts retorqued,778.98 ± 7.97,$(2.45 \pm 0.02) \times 10^{1}$ L/year,"P$_{\infty}$ = 1.013 bar, P$_0$ = 2.736 bar, k..."
2,GTP NPT resealed,GTP NPT resealed,23.29 ± 0.26,$(6.62 \pm 0.07) \times 10^{-1}$ L/year,"P$_{\infty}$ = 1.013 bar, P$_0$ = 2.922 bar, k..."
3,RTL rebent,RTL rebent,13.82 ± 0.30,$(2.64 \pm 0.05) \times 10^{-1}$ L/year,"P$_{\infty}$ = 1.013 bar, P$_0$ = 3.819 bar, k..."
4,TC calibration,TCs calibration,11.72 ± 0.24,$(2.28 \pm 0.04) \times 10^{-1}$ L/year,"P$_{\infty}$ = 1.013 bar, P$_0$ = 3.750 bar, k..."
5,Pump PRV installed,Pump PRV installed,12.28 ± 0.33,$(2.41 \pm 0.06) \times 10^{-1}$ L/year,"P$_{\infty}$ = 1.013 bar, P$_0$ = 3.754 bar, k..."


Fit diagnostics; early high-leak rows use the resolved-range fit and omit the pressure-readout floor from the residuals.


,Dataset,P_inf_bar_abs,RMSE_mbar,first_residual_mbar,last_residual_mbar,max_abs_residual_mbar,note
0,Initial check,1.01325,9.0,3.2,31.2,32.4,fit uses P-P_atm >= 50 mbar; tail below this t...
1,FLM bolts retorqued,1.01325,10.4,22.7,44.3,44.3,fit uses P-P_atm >= 50 mbar; tail below this t...
2,GTP NPT resealed,1.01325,8.4,3.0,2.1,33.6,ok
3,RTL rebent,1.01325,7.8,-8.7,-2.4,25.8,ok
4,TC calibration,1.01325,7.8,2.0,-7.6,27.7,ok
5,Pump PRV installed,1.01325,10.0,-13.1,-45.9,45.9,ok


### High-leak tail definition

For the high-leak checks, the pressure-decay model is kept fixed to atmospheric pressure,
\[
P(t)=P_{\mathrm{atm}} + A e^{kt}, \qquad k<0 .
\]
For each averaged pressure sample, define
\[
\Delta P_i = P_i - P_{\mathrm{atm}} .
\]
The unresolved late-time tail is defined as
\[
\mathcal{T}=\{i:\Delta P_i < \Delta P_{\mathrm{cut}}\},
\qquad \Delta P_{\mathrm{cut}}=50\,\mathrm{mbar} .
\]
The high-leak fits use only
\[
\mathcal{F}=\{i:\Delta P_i \ge \Delta P_{\mathrm{cut}}\},
\]
and the fitted curves are displayed only while
\(P_{\mathrm{fit}}(t)-P_{\mathrm{atm}}\ge \Delta P_{\mathrm{cut}}\).


In [3]:
def draw_system_traces(axis, traces, *, linewidth=2.8, band_alpha=0.18):
    y_samples = []
    for trace in traces:
        result = trace['result']
        y = pressure_above_atm_bar(result.averaged.pressure_abs_bar)
        y_low = pressure_above_atm_bar(result.averaged.pressure_abs_bar - result.averaged.pressure_sigma_bar)
        y_high = pressure_above_atm_bar(result.averaged.pressure_abs_bar + result.averaged.pressure_sigma_bar)
        axis.fill_between(result.averaged.time_h, y_low, y_high, color=trace['color'], alpha=band_alpha)
        axis.plot(result.averaged.time_h, y, color=trace['color'], linewidth=linewidth, label=trace['line_label'])
        fit_curve_time_h, fit_curve_pressure_bar = trace_fit_curve(trace)
        if fit_curve_time_h is not None and fit_curve_pressure_bar is not None and len(fit_curve_time_h) > 0:
            fit_y = pressure_above_atm_bar(fit_curve_pressure_bar)
            axis.plot(
                fit_curve_time_h,
                fit_y,
                color='black',
                linewidth=1.15,
                linestyle=(0, (6, 3)),
                alpha=0.80,
                zorder=6,
            )
            y_samples.append(fit_y)
        y_samples.extend([y_low, y_high])
    return y_samples


def add_manual_points(axis):
    axis.errorbar(
        top_system_time_h,
        pressure_above_atm_bar(top_system_pressure_bar_abs),
        yerr=pressure_above_atm_yerr(top_system_pressure_bar_abs, top_system_sigma_bar),
        fmt='o', capsize=3.0, color='#882255', label='Top of system', markersize=5.8, zorder=7,
    )
    axis.errorbar(
        bottom_system_time_h,
        pressure_above_atm_bar(bottom_system_pressure_bar_abs),
        yerr=pressure_above_atm_yerr(bottom_system_pressure_bar_abs, bottom_system_sigma_bar),
        fmt='+', capsize=3.0, color='#44AA99', label='Bottom of system',
        markersize=10.5, markeredgewidth=2.0, zorder=9,
    )
    axis.errorbar(
        full_system_no_gas_trap_time_h,
        pressure_above_atm_bar(full_system_no_gas_trap_pressure_bar_abs),
        yerr=pressure_above_atm_yerr(full_system_no_gas_trap_pressure_bar_abs, full_system_no_gas_trap_sigma_bar),
        fmt='x', capsize=3.0, color='#332288', label='Without gas trap',
        markersize=9.0, markeredgewidth=2.0, zorder=8,
    )


def style_log_pressure_axis(axis, *, ylabel=True):
    axis.set_xlim(0.0, PLOT_X_MAX_H)
    axis.set_yscale('log')
    axis.set_xlabel('Time (hours)', fontsize=26, labelpad=13)
    if ylabel:
        axis.set_ylabel('Pressure (barg)', fontsize=26)
    axis.tick_params(axis='both', which='major', labelsize=20)
    axis.tick_params(axis='both', which='minor', labelsize=14)
    axis.xaxis.set_major_locator(MultipleLocator(2.5))
    axis.xaxis.set_minor_locator(MultipleLocator(0.5))
    axis.yaxis.set_major_locator(LogLocator(base=10.0, subs=(1.0,), numticks=8))
    axis.yaxis.set_minor_locator(LogLocator(base=10.0, subs=np.arange(2, 10) * 0.1, numticks=80))
    axis.set_axisbelow(True)
    axis.grid(True, which='major', alpha=0.36, linewidth=0.8)
    axis.grid(True, which='minor', alpha=0.18, linewidth=0.55, linestyle=':')


def set_log_y_limits(axis, samples, *, extra_samples=(), lower_factor=1.25, upper_factor=1.15):
    arrays = []
    for sample in [*samples, *extra_samples]:
        sample = np.asarray(sample, dtype=float)
        sample = sample[np.isfinite(sample) & (sample > 0.0)]
        if len(sample) > 0:
            arrays.append(sample)
    if not arrays:
        return
    y_all = np.concatenate(arrays)
    axis.set_ylim(max(MIN_LOG_PRESSURE_BAR, float(np.min(y_all)) / lower_factor), float(np.max(y_all)) * upper_factor)


large_leak_extra_samples = [
    pressure_above_atm_bar(top_system_pressure_bar_abs + top_system_sigma_bar),
    pressure_above_atm_bar(bottom_system_pressure_bar_abs + bottom_system_sigma_bar),
    pressure_above_atm_bar(full_system_no_gas_trap_pressure_bar_abs + full_system_no_gas_trap_sigma_bar),
]

fig_large, axis_large = plt.subplots(figsize=(9.2, 6.8))
large_samples = draw_system_traces(axis_large, system_summary_traces[:3])
add_manual_points(axis_large)
style_log_pressure_axis(axis_large)
set_log_y_limits(axis_large, large_samples, extra_samples=large_leak_extra_samples, lower_factor=1.25, upper_factor=1.20)
axis_large.legend(loc='lower left', fontsize=16, framealpha=0.95, ncol=1)
fig_large.subplots_adjust(left=0.14, right=0.98, top=0.98, bottom=0.14)
display(fig_large)
plt.close(fig_large)

fig_commissioning, axis_commissioning = plt.subplots(figsize=(9.2, 6.8))
commissioning_samples = draw_system_traces(axis_commissioning, system_summary_traces[2:])
style_log_pressure_axis(axis_commissioning)
set_log_y_limits(axis_commissioning, commissioning_samples, lower_factor=1.05, upper_factor=1.08)
axis_commissioning.legend(loc='lower left', fontsize=16, framealpha=0.95, ncol=1)
fig_commissioning.subplots_adjust(left=0.14, right=0.98, top=0.98, bottom=0.14)
display(fig_commissioning)
plt.close(fig_commissioning)


<Figure size 1380x1020 with 1 Axes>

<Figure size 1380x1020 with 1 Axes>

In [4]:
fig_combined, axes_combined = plt.subplots(1, 2, figsize=(14.2, 5.8))

combined_large_samples = draw_system_traces(axes_combined[0], system_summary_traces[:3], linewidth=2.6)
add_manual_points(axes_combined[0])
style_log_pressure_axis(axes_combined[0], ylabel=True)
set_log_y_limits(
    axes_combined[0],
    combined_large_samples,
    extra_samples=large_leak_extra_samples,
    lower_factor=1.25,
    upper_factor=1.20,
)
axes_combined[0].legend(loc='lower left', fontsize=14, framealpha=0.95, ncol=1)

combined_commissioning_samples = draw_system_traces(axes_combined[1], system_summary_traces[2:], linewidth=2.6)
style_log_pressure_axis(axes_combined[1], ylabel=False)
set_log_y_limits(
    axes_combined[1],
    combined_commissioning_samples,
    lower_factor=1.05,
    upper_factor=1.08,
)


RIGHT_PANEL_Y_TICKS = np.array([1.5, 2.0, 2.5, 3.0])


def plain_barg_log_tick(value, _position):
    if np.any(np.isclose(value, RIGHT_PANEL_Y_TICKS, rtol=0.0, atol=1e-8)):
        return f'{value:g}'
    return ''


plain_barg_formatter = FuncFormatter(plain_barg_log_tick)
axes_combined[1].set_yticks(RIGHT_PANEL_Y_TICKS)
axes_combined[1].yaxis.set_major_formatter(plain_barg_formatter)
axes_combined[1].yaxis.set_minor_formatter(FuncFormatter(lambda _value, _position: ''))
axes_combined[1].legend(loc='lower left', fontsize=14, framealpha=0.95, ncol=1)

def combined_left_x_tick(value, _position):
    if 0.0 <= value <= PLOT_X_MAX_H:
        return f'{value:g}'
    return ''


axes_combined[0].xaxis.set_major_formatter(FuncFormatter(combined_left_x_tick))
axes_combined[1].xaxis.set_major_formatter(FuncFormatter(lambda value, _position: f'{value:g}' if 0.0 <= value <= PLOT_X_MAX_H else ''))

fig_combined.subplots_adjust(left=0.09, right=0.99, top=0.98, bottom=0.18, wspace=0.12)
display(fig_combined)
plt.close(fig_combined)


<Figure size 2130x870 with 2 Axes>

In [5]:
VACUUM_PULLDOWN_READING_INHG = -26.0
VACUUM_PULLDOWN_ROOM_TEMP_C = 21.0
COMPOUND_GAUGE_MIN_PSIG = -30.0 * 0.491154
COMPOUND_GAUGE_MAX_PSIG = 60.0
COMPOUND_GAUGE_SPAN_PSIG = COMPOUND_GAUGE_MAX_PSIG - COMPOUND_GAUGE_MIN_PSIG
COMPOUND_GAUGE_RESOLUTION_INHG = 1.0
INHG_TO_PSI = 0.491154
PSI_TO_BAR = leak_tools.INHG_TO_BAR / INHG_TO_PSI


def compound_grade_a_sigma_inhg(reading_inhg, *, resolution_inhg=COMPOUND_GAUGE_RESOLUTION_INHG):
    reading_psig = reading_inhg * INHG_TO_PSI
    quarter_span_psig = 0.25 * COMPOUND_GAUGE_SPAN_PSIG
    in_end_quarter = (
        reading_psig <= COMPOUND_GAUGE_MIN_PSIG + quarter_span_psig
        or reading_psig >= COMPOUND_GAUGE_MAX_PSIG - quarter_span_psig
    )
    accuracy_fraction = 0.02 if in_end_quarter else 0.01
    accuracy_sigma_psig = accuracy_fraction * COMPOUND_GAUGE_SPAN_PSIG / np.sqrt(3.0)
    resolution_sigma_psig = resolution_inhg * INHG_TO_PSI / np.sqrt(3.0)
    return np.sqrt(accuracy_sigma_psig**2 + resolution_sigma_psig**2) / INHG_TO_PSI


def abs_bar_to_inhg(abs_bar):
    return (abs_bar - leak_tools.P_ATM_BAR) / leak_tools.INHG_TO_BAR


WATER_ANTOINE_A = 5.40221
WATER_ANTOINE_B = 1838.675
WATER_ANTOINE_C = -31.737


def water_vapor_pressure_bar_antoine(temp_c):
    temp_k = float(temp_c) + 273.15
    log10_pressure_bar = WATER_ANTOINE_A - WATER_ANTOINE_B / (temp_k + WATER_ANTOINE_C)
    return 10.0 ** log10_pressure_bar


compound_gauge_reading_psig = VACUUM_PULLDOWN_READING_INHG * INHG_TO_PSI
compound_gauge_quarter_span_psig = 0.25 * COMPOUND_GAUGE_SPAN_PSIG
compound_gauge_in_end_quarter = (
    compound_gauge_reading_psig <= COMPOUND_GAUGE_MIN_PSIG + compound_gauge_quarter_span_psig
    or compound_gauge_reading_psig >= COMPOUND_GAUGE_MAX_PSIG - compound_gauge_quarter_span_psig
)
compound_gauge_accuracy_fraction = 0.02 if compound_gauge_in_end_quarter else 0.01
compound_gauge_accuracy_half_width_mbar = (
    compound_gauge_accuracy_fraction * COMPOUND_GAUGE_SPAN_PSIG * PSI_TO_BAR * 1000.0
)
compound_gauge_accuracy_sigma_mbar = compound_gauge_accuracy_half_width_mbar / np.sqrt(3.0)
compound_gauge_resolution_half_width_mbar = COMPOUND_GAUGE_RESOLUTION_INHG * leak_tools.INHG_TO_BAR * 1000.0
compound_gauge_resolution_sigma_mbar = compound_gauge_resolution_half_width_mbar / np.sqrt(3.0)

vacuum_pulldown_sigma_inhg = compound_grade_a_sigma_inhg(VACUUM_PULLDOWN_READING_INHG)
vacuum_pulldown_abs_bar = leak_tools.P_ATM_BAR + VACUUM_PULLDOWN_READING_INHG * leak_tools.INHG_TO_BAR
vacuum_pulldown_sigma_bar = vacuum_pulldown_sigma_inhg * leak_tools.INHG_TO_BAR
hfe_only_bar = leak_tools.hfe_vapor_pressure_bar(VACUUM_PULLDOWN_ROOM_TEMP_C)
water_only_bar = water_vapor_pressure_bar_antoine(VACUUM_PULLDOWN_ROOM_TEMP_C)
hfe_plus_water_bar = hfe_only_bar + water_only_bar

vacuum_pulldown_abs_mbar = 1000.0 * vacuum_pulldown_abs_bar
vacuum_pulldown_sigma_mbar = 1000.0 * vacuum_pulldown_sigma_bar
hfe_only_mbar = 1000.0 * hfe_only_bar
water_only_mbar = 1000.0 * water_only_bar
hfe_plus_water_mbar = 1000.0 * hfe_plus_water_bar

pulldown_cases = [
    {
        'case': 'Measured residual pressure',
        'pressure_mbar_abs': vacuum_pulldown_abs_mbar,
        'gauge_inhg': VACUUM_PULLDOWN_READING_INHG,
    },
    {
        'case': 'HFE-7200 vapor pressure',
        'pressure_mbar_abs': hfe_only_mbar,
        'gauge_inhg': abs_bar_to_inhg(hfe_only_bar),
    },
    {
        'case': 'HFE-7200 + water vapor',
        'pressure_mbar_abs': hfe_plus_water_mbar,
        'gauge_inhg': abs_bar_to_inhg(hfe_plus_water_bar),
    },
    {
        'case': 'Water vapor only',
        'pressure_mbar_abs': water_only_mbar,
        'gauge_inhg': abs_bar_to_inhg(water_only_bar),
    },
]

pulldown_comparison_rows = []
for case in pulldown_cases:
    pressure_difference_mbar = case['pressure_mbar_abs'] - vacuum_pulldown_abs_mbar
    pulldown_comparison_rows.append(
        {
            'case': case['case'],
            'pressure [mbar abs]': case['pressure_mbar_abs'],
            'equivalent gauge reading [inHg]': case['gauge_inhg'],
            'difference from measured [mbar]': pressure_difference_mbar,
            'difference from measured [sigma]': pressure_difference_mbar / vacuum_pulldown_sigma_mbar,
        }
    )
pulldown_comparison = pd.DataFrame(pulldown_comparison_rows)

fig, axis = plt.subplots(figsize=(7.8, 6.2))

x_expected = 0.0
x_measured = 1.05
bar_width = 0.58
hfe_color = '#009E73'
water_color = '#56B4E9'
measured_color = '#F2B880'
uncertainty_color = '#3A2618'

axis.bar(
    x_expected,
    hfe_only_mbar,
    width=bar_width,
    color=hfe_color,
    edgecolor='black',
    linewidth=0.8,
    label='HFE-7200 vapor pressure',
    zorder=3,
)
axis.bar(
    x_expected,
    water_only_mbar,
    bottom=hfe_only_mbar,
    width=bar_width,
    color=water_color,
    edgecolor='black',
    linewidth=0.8,
    label='Water vapor pressure',
    zorder=3,
)
axis.bar(
    x_measured,
    vacuum_pulldown_abs_mbar,
    width=bar_width,
    color=measured_color,
    edgecolor='black',
    linewidth=0.8,
    yerr=vacuum_pulldown_sigma_mbar,
    capsize=9,
    error_kw={
        'ecolor': uncertainty_color,
        'elinewidth': 2.3,
        'capthick': 2.3,
        'zorder': 5,
    },
    label='Measured residual pressure',
    zorder=3,
)

axis.text(
    x_expected,
    0.5 * hfe_only_mbar,
    f'HFE-7200\n{hfe_only_mbar:.1f} mbar',
    ha='center',
    va='center',
    color='white',
    fontsize=14,
    fontweight='bold',
)
axis.text(
    x_expected,
    hfe_only_mbar + 0.5 * water_only_mbar,
    f'water\n{water_only_mbar:.1f} mbar',
    ha='center',
    va='center',
    color='#0E3F5A',
    fontsize=12.5,
    fontweight='bold',
)
axis.text(
    x_expected,
    hfe_plus_water_mbar + 6.0,
    f'total {hfe_plus_water_mbar:.1f} mbar',
    ha='center',
    va='bottom',
    color='0.20',
    fontsize=13,
)
axis.text(
    x_measured,
    vacuum_pulldown_abs_mbar + vacuum_pulldown_sigma_mbar + 6.0,
    f'{vacuum_pulldown_abs_mbar:.1f} $\\pm$ {vacuum_pulldown_sigma_mbar:.1f} mbar',
    ha='center',
    va='bottom',
    color='0.20',
    fontsize=13,
)

axis.set_xlim(-0.55, 1.60)
axis.set_ylim(0.0, 210.0)
axis.set_ylabel('Absolute pressure [mbar]', fontsize=18)
axis.set_xticks([x_expected, x_measured])
axis.set_xticklabels(['Expected', 'Measured'], fontsize=15)
axis.tick_params(axis='y', labelsize=14)
axis.yaxis.set_major_locator(MultipleLocator(50.0))
axis.yaxis.set_minor_locator(MultipleLocator(10.0))
axis.grid(True, axis='y', which='major', alpha=0.28, linewidth=0.8)
axis.grid(True, axis='y', which='minor', alpha=0.12, linewidth=0.5, linestyle=':')
axis.spines['top'].set_visible(False)
axis.spines['right'].set_visible(False)
axis.legend(
    loc='lower center',
    bbox_to_anchor=(0.5, 1.02),
    ncol=3,
    frameon=True,
    framealpha=0.95,
    fontsize=11.8,
    columnspacing=1.2,
    handlelength=1.4,
)

fig.subplots_adjust(left=0.14, right=0.99, top=0.86, bottom=0.18)
display(fig)
plt.close(fig)

display(
    pulldown_comparison.round(
        {
            'pressure [mbar abs]': 1,
            'equivalent gauge reading [inHg]': 2,
            'difference from measured [mbar]': 1,
            'difference from measured [sigma]': 2,
        }
    )
)

print(
    'Measured pulldown from the -30 inHg to 60 psi Grade A compound gauge: '
    f'{VACUUM_PULLDOWN_READING_INHG:.1f} +/- {vacuum_pulldown_sigma_inhg:.2f} inHg '
    f'(1 sigma), or {vacuum_pulldown_abs_mbar:.1f} +/- {vacuum_pulldown_sigma_mbar:.1f} mbar abs.'
)
print(
    'Gauge uncertainty model: the -26 inHg reading is in the end quarter of the compound-gauge scale, '
    f'so the Grade A accuracy term is +/- {compound_gauge_accuracy_half_width_mbar:.1f} mbar as a rectangular tolerance '
    f'(sigma = {compound_gauge_accuracy_sigma_mbar:.1f} mbar). '
    f'The assumed reading resolution is +/- {compound_gauge_resolution_half_width_mbar:.1f} mbar '
    f'(sigma = {compound_gauge_resolution_sigma_mbar:.1f} mbar). '
    f'Added in quadrature, this gives sigma = {vacuum_pulldown_sigma_mbar:.1f} mbar.'
)
print(f'Room-temperature assumption for this comparison: {VACUUM_PULLDOWN_ROOM_TEMP_C:.1f} C.')
print(
    f'HFE-7200 saturation at that temperature is {hfe_only_mbar:.1f} mbar abs '
    f'({abs_bar_to_inhg(hfe_only_bar):.2f} inHg gauge).'
)
print(
    f'Water saturation at that temperature is {water_only_mbar:.1f} mbar abs '
    f'({abs_bar_to_inhg(water_only_bar):.2f} inHg gauge).'
)
print(
    'If both condensed HFE-7200 and water were present, an idealized upper-bound '
    f'saturation pressure is {hfe_plus_water_mbar:.1f} mbar abs '
    f'({abs_bar_to_inhg(hfe_plus_water_bar):.2f} inHg gauge).'
)


<Figure size 1170x930 with 1 Axes>

,case,pressure [mbar abs],equivalent gauge reading [inHg],difference from measured [mbar],difference from measured [sigma]
0,Measured residual pressure,132.8,-26.00,0.0,0.00
1,HFE-7200 vapor pressure,137.2,-25.87,4.4,0.07
2,HFE-7200 + water vapor,162.1,-25.14,29.3,0.47
3,Water vapor only,24.9,-29.19,-107.9,-1.72


Measured pulldown from the -30 inHg to 60 psi Grade A compound gauge: -26.0 +/- 1.85 inHg (1 sigma), or 132.8 +/- 62.6 mbar abs.
Gauge uncertainty model: the -26 inHg reading is in the end quarter of the compound-gauge scale, so the Grade A accuracy term is +/- 103.1 mbar as a rectangular tolerance (sigma = 59.5 mbar). The assumed reading resolution is +/- 33.9 mbar (sigma = 19.6 mbar). Added in quadrature, this gives sigma = 62.6 mbar.
Room-temperature assumption for this comparison: 21.0 C.
HFE-7200 saturation at that temperature is 137.2 mbar abs (-25.87 inHg gauge).
Water saturation at that temperature is 24.9 mbar abs (-29.19 inHg gauge).
If both condensed HFE-7200 and water were present, an idealized upper-bound saturation pressure is 162.1 mbar abs (-25.14 inHg gauge).
